# Differential expression from ARCHS4One DESeq2 table per study, written to `data/rnaseq_archs6/`.

In [24]:
import pathlib
import re

import archs4py as a4
import h5py
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

`CONTRASTS` is the whole configuration: per study the file label, the characteristic holding thegroups, the values that make up each arm, and any covariates. `case` and `control` take one value ora list of them; `regex=True` matches with `re.fullmatch` instead, and `field` may name a samplemetadata field such as `title` rather than a characteristic.

In [25]:
DATA_DIR = pathlib.Path("../../data")
ARCHS4_DIR = DATA_DIR / "archs4"
ARCHS4_VERSION = "latest"
ARCHS4_FILE = ARCHS4_DIR / f"human_gene_v2.{ARCHS4_VERSION}.h5"
BUILD_DIR = pathlib.Path("../../build")
OUTPUT_DIR = BUILD_DIR / "rnaseq_archs6"
QC_LOG_FILE = OUTPUT_DIR / "qc_log.csv"

MIN_ALIGNED_READS = 5_000_000
MAX_SINGLE_CELL_PROBABILITY = 0.5
MIN_GROUP_SIZE = 3
MIN_LOG_CPM = 1

WP4_DATASETS = [
    "GSE95587",
    "GSE147507",
    "GSE174745",
    "GSE179923",
    "GSE152418",
    "GSE251849",
    "GSE123496",
    "GSE136666",
    "GSE216281",
    "GSE68719",
]

CONTRASTS = {
    "GSE236562": dict(
        label="COVID19_brain_GSE236562",
        field="covid status",
        case="COVID",
        control="CONTROL",
        covariates=["alzheimers status", "brain region"],
    ),
    "GSE188847": dict(
        label="COVID19_brain_GSE188847",
        field="group",
        case="COVID-19",
        control="Control",
    ),
    "GSE164332": dict(
        label="COVID19_brain_GSE164332",
        field="disease",
        case="COVID-19",
        control="Control",
    ),
    "GSE157103": dict(
        label="COVID19_leukocytes_GSE157103",
        field="disease state",
        case="COVID-19",
        control="non-COVID-19",
    ),
    "GSE205450": dict(
        label="PD_striatum_GSE205450",
        field="disease",
        case="PD",
        control="Control",
        covariates=["region"],
    ),
    "GSE95587": dict(
        label="AD_brain_GSE95587",
        field="diagnosis",
        case="Alzheimer's disease",
        control="control",
    ),
    "GSE53697": dict(
        label="AD_brain_GSE53697",
        field="disease status",
        case="advanced Alzheimer's Disease",
        control="control",
    ),
    "GSE184942": dict(
        label="AD_brain_GSE184942", field="tissue", case="AD", control="Health"
    ),
    "GSE193438": dict(
        label="AD_brain_GSE193438",
        field="disease",
        case="AD",
        control="Helathy Control",
    ),
    "GSE123496": dict(
        label="MS_brain_GSE123496",
        field="disease state",
        case="MS",
        control="healthy control",
        covariates=["tissue"],
    ),
    "GSE138614": dict(
        label="MS_brain_GSE138614",
        field="diagnosis",
        case="Multiple sclerosis",
        control="Control",
    ),
    "GSE68719": dict(
        label="PD_brain_GSE68719",
        field="title",
        regex=True,
        case=r"P_\d+",
        control=r"C_\d+ \[reanalysis of GSM\d+\]",
    ),
    "GSE136666": dict(
        label="PD_brain_GSE136666",
        field="disease state",
        case="Parkinson\u00e2\u0080\u0099s disease",
        control="Control",
        covariates=["tissue"],
    ),
    "GSE135036": dict(
        label="PD_brain_GSE135036",
        field="diagnosis",
        case="Parkinson\u00e2\u0080\u0099s disease (PD)",
        control="Control",
    ),
    "GSE216281": dict(
        label="PD_brain_GSE216281",
        field="braak lewy_body_stage",
        case=["4", "5", "6"],
        control="0",
    ),
    "GSE147507": dict(
        label="COVID19_cl_GSE147507",
        field="source_name_ch1",
        regex=True,
        case=r"SARS-CoV-2 infected .*",
        control=r"Mock treated .*",
    ),
    "GSE152418": dict(
        label="COVID19_pbmc_GSE152418",
        field="disease state",
        case="COVID-19",
        control="Healthy",
    ),
    "GSE251849": dict(
        label="COVID19_pbmc_GSE251849",
        field="group",
        case="Long COVID",
        control="Healthy",
    ),
    "GSE224615": dict(
        label="COVID19_pbmc_GSE224615", field="lc status", case="LC", control="Non-LC"
    ),
    "GSE174745": dict(
        label="COVID19_dopaminergic_GSE174745",
        field="viral infected",
        case="SARS-Cov-2 infected 48 hours",
        control="no",
    ),
    "GSE179923": dict(
        label="COVID19_hipsc_GSE179923",
        field="treatment",
        case="SARS-CoV-2",
        control="Mock",
    ),
    "GSE157852": dict(
        label="COVID19_organoid_GSE157852",
        field="treatment",
        case="SARS-CoV-2 72 hpi",
        control="Mock 72 hpi",
    ),
    "GSE166190": dict(
        label="COVID19_blood_GSE166190",
        field="sars-cov-2 infection",
        case="Positive",
        control="Negative",
    ),
    "GSE169687": dict(
        label="COVID19_blood_GSE169687",
        field="disease severity",
        case=["Mild", "Moderate", "Severe", "Critical"],
        control="Healthy",
    ),
    "GSE217948": dict(
        label="COVID19_blood_GSE217948",
        field="title",
        regex=True,
        case=r"Whole blood, COVID positive, rep \d+",
        control=r"Whole blood, healthy control, rep \d+",
    ),
    "GSE104704": dict(
        label="aging_brain_GSE104704",
        field="study group",
        case=["Old", "Aged"],
        control="Young",
    ),
    "GSE159699": dict(
        label="aging_brain_GSE159699", field="age", case="old", control="young"
    ),
}

Download the counts file if it is not there yet (62 GB, once).

In [26]:
ARCHS4_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not ARCHS4_FILE.exists():
    a4.download.counts("human", path=str(ARCHS4_DIR), version=ARCHS4_VERSION)

In [27]:
h5 = h5py.File(ARCHS4_FILE, "r")


def read_strings(path):
    return np.array([v.decode() if isinstance(v, bytes) else v for v in h5[path][:]])


series_ids = read_strings("meta/samples/series_id")
sample_ids = read_strings("meta/samples/geo_accession")
gene_ids = read_strings("meta/genes/ensembl_gene")
aligned_reads = h5["meta/samples/alignedreads"][:]
single_cell_probabilities = h5["meta/samples/singlecellprobability"][:]

The sample table of a study, carrying only the characteristics its contrast names, plus a tissue for the summary.

In [28]:
def make_column_name(name):
    return re.sub(r"\W+", "_", name).strip("_").lower()


def get_characteristic(characteristics, field):
    match = re.search(rf"(?:^|,)\s*{re.escape(field)}:\s*([^,]*)", characteristics)
    return match.group(1).strip() if match else None


def get_field(field, indices, characteristics):
    if field in h5["meta/samples"]:
        return [h5[f"meta/samples/{field}"][i].decode() for i in indices]
    return [get_characteristic(c, field) for c in characteristics]


def get_tissue(characteristics, source_name):
    for field in ("tissue", "tissue/cell type", "cell type", "tissue subtype"):
        value = get_characteristic(characteristics, field)
        if value:
            return value
    return source_name


def load_samples(gse, contrast):
    indices = np.flatnonzero(np.char.find(series_ids, gse) >= 0)
    characteristics = [
        h5["meta/samples/characteristics_ch1"][i].decode() for i in indices
    ]
    fields = [contrast["field"], *contrast.get("covariates", [])]
    samples = pd.DataFrame(
        {make_column_name(f): get_field(f, indices, characteristics) for f in fields},
        index=sample_ids[indices],
    )
    samples["tissue_label"] = [
        get_tissue(c, h5["meta/samples/source_name_ch1"][i].decode())
        for c, i in zip(characteristics, indices)
    ]
    samples["index"] = indices
    samples["aligned_reads"] = aligned_reads[indices]
    samples["single_cell_probability"] = single_cell_probabilities[indices]
    return samples

A sample joins an arm when its value is one of the arm's values, or matches its pattern under `regex`.

In [29]:
def match_group(values, spec, regex):
    if regex:
        return values.str.fullmatch(spec).fillna(False)
    return values.isin([spec] if isinstance(spec, str) else spec)


def assign_groups(samples, contrast):
    values = samples[make_column_name(contrast["field"])].astype("string")
    regex = contrast.get("regex", False)
    groups = pd.Series(None, index=samples.index, dtype=object)
    groups[match_group(values, contrast["control"], regex)] = "control"
    groups[match_group(values, contrast["case"], regex)] = "case"
    return samples.assign(group=groups)

Every contrast at a glance, to check the chosen field and arms against what the study actually holds.

In [30]:
def summarise_contrast(gse, contrast):
    samples = load_samples(gse, contrast)
    values = samples[make_column_name(contrast["field"])].value_counts(dropna=False)
    samples = assign_groups(samples, contrast)
    tissues = sorted(samples["tissue_label"].dropna().unique())
    counts = samples["group"].value_counts()
    return {
        "dataset": contrast["label"],
        "in WP4 datasets": "yes" if gse in WP4_DATASETS else "no",
        "tissue": " / ".join(tissues)
        if len(tissues) <= 3
        else f"{len(tissues)} tissues",
        "field": contrast["field"],
        "values": ", ".join(f"{v}: {n}" for v, n in values.items()),
        "case": contrast["case"],
        "control": contrast["control"],
        "n_case": counts.get("case", 0),
        "n_control": counts.get("control", 0),
        "n_excluded": samples["group"].isna().sum(),
        "n_total": len(samples),
    }


summary = pd.DataFrame([summarise_contrast(gse, c) for gse, c in CONTRASTS.items()])
display(HTML(f'<div class="mathjax_ignore">{summary.to_html()}</div>'))

,dataset,in WP4 datasets,tissue,field,values,case,control,n_case,n_control,n_excluded,n_total
0,COVID19_brain_GSE236562,no,Brain,covid status,"CONTROL: 16, COVID: 16",COVID,CONTROL,16,16,0,32
1,COVID19_brain_GSE188847,no,Frontal cortex of brain / Primary neuron,group,"Control: 25, COVID-19: 22, Young: 10, ICU/VENT: 9, Old: 6, IFNB-high: 3, IFNB-low: 3, IFNG-high: 3, IFNG-low: 3, TNF-high: 2, TNF-low: 2",COVID-19,Control,22,25,41,88
2,COVID19_brain_GSE164332,no,Frontal cortex,disease,"COVID-19: 9, Control: 7",COVID-19,Control,9,7,0,16
3,COVID19_leukocytes_GSE157103,no,leukocyte,disease state,"COVID-19: 100, non-COVID-19: 26",COVID-19,non-COVID-19,100,26,0,126
4,PD_striatum_GSE205450,no,Post-mortem brain tissue,disease,"Control: 81, PD: 69",PD,Control,69,81,0,150
5,AD_brain_GSE95587,yes,fusiform gyrus,diagnosis,"Alzheimer's disease: 83, control: 33",Alzheimer's disease,control,83,33,0,116
6,AD_brain_GSE53697,no,brain,disease status,"advanced Alzheimer's Disease: 9, control: 8",advanced Alzheimer's Disease,control,9,8,0,17
7,AD_brain_GSE184942,no,AD / Health,tissue,"Health: 5, AD: 5",AD,Health,5,5,0,10
8,AD_brain_GSE193438,no,4 tissues,disease,"AD: 16, Helathy Control: 16, LBD: 16",AD,Helathy Control,16,16,16,48
9,MS_brain_GSE123496,yes,5 tissues,disease state,"MS: 25, healthy control: 25",MS,healthy control,25,25,0,50


Its counts, indexed by Ensembl gene id.

In [8]:
def load_counts(samples):
    indices = samples["index"].to_numpy()
    order = np.argsort(indices)
    counts = h5["data/expression"][:, indices[order]][:, np.argsort(order)]
    return pd.DataFrame(counts, index=gene_ids, columns=samples.index)

Sample QC: single-cell probability, then depth, then whether the sample is in the contrast at all.

In [9]:
def apply_qc(samples):
    reasons = pd.Series("kept", index=samples.index)
    reasons[samples["single_cell_probability"] > MAX_SINGLE_CELL_PROBABILITY] = (
        "single cell"
    )
    reasons[samples["aligned_reads"] < MIN_ALIGNED_READS] = "shallow"
    reasons[samples["group"].isna()] = "not in contrast"
    return samples.assign(qc=reasons)

Genes expressed above `MIN_LOG_CPM` in at least as many samples as the smaller group has.

In [10]:
def filter_genes(counts, samples):
    log_cpm = np.log2(counts / counts.sum() * 1e6 + 1)
    return counts[
        (log_cpm > MIN_LOG_CPM).sum(axis=1) >= samples["group"].value_counts().min()
    ]

DESeq2, with the contrast's covariates in the design formula.

In [11]:
def run_deseq2(counts, samples, contrast):
    covariates = [
        make_column_name(covariate) for covariate in contrast.get("covariates", [])
    ]
    covariates = [c for c in covariates if samples[c].nunique() > 1]
    design = "~ " + " + ".join(covariates + ["group"])
    dds = DeseqDataSet(
        counts=counts.T,
        metadata=samples[covariates + ["group"]],
        design=design,
        quiet=True,
    )
    dds.deseq2()
    stats = DeseqStats(dds, contrast=["group", "case", "control"], quiet=True)
    stats.summary()
    return stats.results_df

Every contrast, writing one file per study and one QC log for all of them.

In [12]:
qc_logs = []
for gse, contrast in CONTRASTS.items():
    samples = apply_qc(assign_groups(load_samples(gse, contrast), contrast))
    qc_logs.append(
        samples.assign(gse=gse)[
            ["gse", "group", "aligned_reads", "single_cell_probability", "qc"]
        ]
    )
    kept = samples[samples["qc"] == "kept"]
    sizes = kept["group"].value_counts()
    if len(sizes) < 2 or sizes.min() < MIN_GROUP_SIZE:
        print(f"{gse}: skipped, {sizes.to_dict()}")
        continue
    counts = filter_genes(load_counts(kept), kept)
    results = run_deseq2(counts, kept, contrast)
    results.to_csv(
        OUTPUT_DIR / f"deseqobj_{contrast['label']}.csv", sep="\t", index_label=False
    )
    print(f"{gse}: {len(counts)} genes, {sizes.to_dict()}")

pd.concat(qc_logs).to_csv(QC_LOG_FILE)

GSE236562: 25505 genes, {'control': 16, 'case': 16}


GSE188847: 23828 genes, {'control': 25, 'case': 22}


GSE164332: 31195 genes, {'case': 7, 'control': 4}


GSE157103: 16852 genes, {'case': 100, 'control': 26}


GSE205450: 19086 genes, {'control': 81, 'case': 69}


...

GSE166190: 26438 genes, {'case': 82, 'control': 15}


GSE169687: 28745 genes, {'case': 137, 'control': 14}


GSE217948: 27833 genes, {'case': 374, 'control': 68}


GSE104704: 25074 genes, {'case': 22, 'control': 8}


GSE159699: 23228 genes, {'case': 10, 'control': 8}
